# Robust Detection of AI-Generated Images — Interactive Analysis

This notebook lets you interactively explore:
- Trained model performance
- Distribution shift experiments
- Saliency maps for individual images
- Confidence analysis

**Run `python run_experiments.py` first** to train the model and generate results.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from pathlib import Path

import config
from config import DEVICE, CHECKPOINT_DIR, LOGS_DIR, PLOTS_DIR

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 1. Load Trained Model

In [ ]:
from evaluate import load_model
import os

checkpoint_path = os.path.join(CHECKPOINT_DIR, f'best_{config.MODEL_TYPE}.pt')
model, model_type = load_model(checkpoint_path)
print(f'Model: {model_type}')

## 2. View Training Curves

In [ ]:
# Load the most recent training log
logs = sorted(Path(LOGS_DIR).glob('*_training.csv'))
if logs:
    df = pd.read_csv(logs[-1])
    print(df.to_string(index=False))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(df['epoch'], df['train_loss'], label='Train')
    ax1.plot(df['epoch'], df['val_loss'],   label='Val')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
    
    ax2.plot(df['epoch'], df['train_acc'], label='Train')
    ax2.plot(df['epoch'], df['val_acc'],   label='Val')
    ax2.set_title('Accuracy'); ax2.legend(); ax2.set_ylim(0,1); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('No training log found. Run train.py first.')

## 3. Evaluation Results Table

In [ ]:
eval_logs = sorted(Path(LOGS_DIR).glob('*_eval.csv'))
if eval_logs:
    df = pd.read_csv(eval_logs[-1])
    # Show key columns nicely
    cols = ['experiment', 'accuracy', 'precision', 'recall', 'f1', 'auc']
    available = [c for c in cols if c in df.columns]
    display(df[available].style.highlight_max(subset=['accuracy','f1','auc'], color='lightgreen')
                               .highlight_min(subset=['accuracy','f1','auc'], color='lightsalmon')
                               .format({c: '{:.4f}' for c in available if c != 'experiment'}))
else:
    print('No eval log found. Run evaluate.py first.')

## 4. Interactive Saliency Map Explorer

In [ ]:
from data.dataset import CIFAKEDataset, get_base_transform, apply_jpeg_compression
from utils.visualization import compute_vanilla_saliency

dataset = CIFAKEDataset(split='test', num_samples=200)
transform = get_base_transform()

def show_saliency(idx: int, jpeg_quality: int = None):
    """
    Show saliency map for image at index `idx`.
    Optionally apply JPEG compression first.
    """
    img_path   = dataset.image_paths[idx]
    true_label = dataset.labels[idx]
    
    pil_img = Image.open(img_path).convert('RGB')
    if jpeg_quality:
        pil_img = apply_jpeg_compression(pil_img, jpeg_quality)
    
    tensor = transform(pil_img).unsqueeze(0)
    
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)
        pred  = probs.argmax(dim=1).item()
    
    saliency = compute_vanilla_saliency(model, tensor, target_class=pred)
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    img_np = np.array(pil_img)
    axes[0].imshow(img_np);                         axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(saliency, cmap='hot');            axes[1].set_title('Saliency Map'); axes[1].axis('off')
    axes[2].imshow(img_np)
    axes[2].imshow(saliency, alpha=0.5, cmap='hot'); axes[2].set_title('Overlay'); axes[2].axis('off')
    
    true_str = 'Real' if true_label==0 else 'Fake'
    pred_str = 'Real' if pred==0 else 'Fake'
    correct  = '✓' if true_label==pred else '✗'
    conf     = probs[0, pred].item()
    title = f'{correct} True: {true_str} | Pred: {pred_str} (conf={conf:.3f})'
    if jpeg_quality:
        title += f' | JPEG q={jpeg_quality}'
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()

# Try first 3 images
for i in range(3):
    show_saliency(i)

## 5. Saliency: Before vs After JPEG Compression

In [ ]:
# Compare saliency maps for same image at different JPEG qualities
idx = 5  # Change this to any index

qualities = [95, 50, 10]
img_path   = dataset.image_paths[idx]
true_label = dataset.labels[idx]
true_str   = 'Real' if true_label==0 else 'Fake'

fig, axes = plt.subplots(len(qualities), 3, figsize=(10, len(qualities)*3))
fig.suptitle(f'Saliency Maps Under Compression (True: {true_str})', fontsize=12)

for row, quality in enumerate(qualities):
    pil_img = Image.open(img_path).convert('RGB')
    pil_img = apply_jpeg_compression(pil_img, quality)
    tensor  = transform(pil_img).unsqueeze(0)
    
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)
        pred  = probs.argmax(dim=1).item()
    
    saliency  = compute_vanilla_saliency(model, tensor, target_class=pred)
    img_np    = np.array(pil_img)
    pred_str  = 'Real' if pred==0 else 'Fake'
    correct   = '✓' if true_label==pred else '✗'
    conf      = probs[0, pred].item()
    
    axes[row,0].imshow(img_np);                          axes[row,0].set_title(f'JPEG q={quality}'); axes[row,0].axis('off')
    axes[row,1].imshow(saliency, cmap='hot');             axes[row,1].set_title('Saliency'); axes[row,1].axis('off')
    axes[row,2].imshow(img_np)
    axes[row,2].imshow(saliency, alpha=0.5, cmap='hot')
    axes[row,2].set_title(f'{correct} Pred:{pred_str} ({conf:.2f})')
    axes[row,2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'saliency_compression_comparison.png'), dpi=150)
plt.show()

## 6. Full Compression Shift Experiment (quick)

In [ ]:
from data.dataset import get_dataloader
from utils.metrics import evaluate_model, performance_drop

print('Running compression shift experiments...')
results = {}

for q in config.JPEG_QUALITY_LEVELS:
    loader  = get_dataloader('test', jpeg_quality=q, num_samples=500, batch_size=16)
    metrics = evaluate_model(model, loader, DEVICE)
    results[q] = metrics
    print(f'  q={q:3d}: AUC={metrics["auc"]:.4f}  acc={metrics["accuracy"]:.4f}  f1={metrics["f1"]:.4f}')

baseline_loader  = get_dataloader('test', num_samples=500, batch_size=16)
baseline_metrics = evaluate_model(model, baseline_loader, DEVICE)
print(f'\nBaseline: AUC={baseline_metrics["auc"]:.4f}  acc={baseline_metrics["accuracy"]:.4f}')

worst_q = min(results, key=lambda q: results[q]['auc'])
print(f'Worst shift: JPEG q={worst_q}, drop={performance_drop(baseline_metrics, results[worst_q]):.4f}')

## 7. Save All Figures for Paper

In [ ]:
from utils.visualization import (
    plot_compression_robustness, 
    plot_robustness_comparison,
    plot_confusion_matrix
)

all_results = {'Baseline': baseline_metrics}
for q, m in results.items():
    all_results[f'JPEG q={q}'] = m

plot_compression_robustness(config.JPEG_QUALITY_LEVELS, results)
plot_robustness_comparison(all_results, metric='auc')
plot_confusion_matrix(baseline_metrics['confusion_matrix'])

print(f'\nAll figures saved to: {PLOTS_DIR}')
print('Files:', os.listdir(PLOTS_DIR))